# 🦴 OsteoVision - YOLOv8 Pose 訓練パイプライン

**このノートブックでやること：**
1. ライブラリのインストール
2. Google Drive をマウント（データ・モデルの保存先）
3. 合成DRR（模擬X線）データを生成
4. YOLOv8 Pose モデルを訓練（GPU使用）
5. 訓練済みモデルを Drive に保存

---
**上のメニューから `ランタイム → ランタイムのタイプを変更 → T4 GPU` を選択してから実行してください。**

## STEP 1: ライブラリのインストール

In [ ]:
!pip install ultralytics opencv-python-headless numpy scipy pydicom pandas tqdm -q
print('✅ インストール完了')

## STEP 2: Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/OsteoVision'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ 保存先: {SAVE_DIR}')

## STEP 3: 合成DRRデータを生成

CTデータなしで合成骨ボリュームから模擬X線画像を自動生成します。  
生成される画像数: **約720枚**（LAT/AP × 金属あり/なし × 各角度）

In [ ]:
import os
import cv2
import numpy as np
import math
import random
import hashlib
import json
import pandas as pd
from scipy.ndimage import affine_transform
from tqdm import tqdm

def get_rotation_matrix(rx_deg, ry_deg, rz_deg):
    rx, ry, rz = math.radians(rx_deg), math.radians(ry_deg), math.radians(rz_deg)
    Rx = np.array([[1,0,0],[0,math.cos(rx),-math.sin(rx)],[0,math.sin(rx),math.cos(rx)]])
    Ry = np.array([[math.cos(ry),0,math.sin(ry)],[0,1,0],[-math.sin(ry),0,math.cos(ry)]])
    Rz = np.array([[math.cos(rz),-math.sin(rz),0],[math.sin(rz),math.cos(rz),0],[0,0,1]])
    return Rz @ Ry @ Rx

def create_synthetic_bone_with_landmarks(size=128, add_metal_implant=False):
    volume = np.zeros((size, size, size), dtype=np.float32)
    femur_shaft_center   = (int(size*0.75), int(size*0.5),      int(size*0.5))
    medial_condyle_center= (int(size*0.3125), int(size*0.546875), int(size*0.65625))
    lateral_condyle_center=(int(size*0.3125), int(size*0.390625), int(size*0.34375))
    tibia_plateau_center = (int(size*0.125), int(size*0.5),      int(size*0.5))
    bone_val  = 1000
    metal_val = 4000 if add_metal_implant else bone_val
    shaft_cx, shaft_cy = int(size*0.5), int(size*0.5)
    shaft_z_start  = int(size*0.375)
    condyle_z_start= int(size*0.25)
    condyle_z_end  = shaft_z_start
    for z in range(size):
        if z > condyle_z_end:
            for y in range(size):
                for x in range(size):
                    if (x-shaft_cx)**2+(y-shaft_cy)**2 <= 12**2:
                        volume[z,y,x] = bone_val
        elif condyle_z_start < z <= condyle_z_end:
            for y in range(size):
                for x in range(size):
                    m = (x-medial_condyle_center[2])**2+(y-medial_condyle_center[1])**2
                    if m <= 14**2:
                        volume[z,y,x] = metal_val if (add_metal_implant and z<40 and m>10**2) else bone_val
                    l = (x-lateral_condyle_center[2])**2+(y-lateral_condyle_center[1])**2
                    if l <= 12**2:
                        volume[z,y,x] = metal_val if (add_metal_implant and z<40 and l>8**2) else bone_val
        elif z <= condyle_z_start:
            for y in range(size):
                for x in range(size):
                    if (x-shaft_cx)**2+(y-shaft_cy)**2 <= 16**2:
                        volume[z,y,x] = bone_val
    landmarks_3d = {
        'femur_shaft':     femur_shaft_center,
        'medial_condyle':  medial_condyle_center,
        'lateral_condyle': lateral_condyle_center,
        'tibia_plateau':   tibia_plateau_center
    }
    return volume, landmarks_3d

def project_3d_point_to_2d(point_3d, rot_matrix, center, out_shape, volume_shape):
    p = np.array(point_3d, dtype=np.float64)
    p_rot = rot_matrix.dot(p - center) + center
    scale_y = out_shape[0] / volume_shape[0]
    scale_x = out_shape[1] / volume_shape[1]
    return (int(p_rot[1] * scale_x), int(p_rot[0] * scale_y))

def convert_to_yolov8_pose(points_2d, img_w, img_h):
    pts = np.array(list(points_2d.values()))
    min_x, max_x = np.min(pts[:,0]), np.max(pts[:,0])
    min_y, max_y = np.min(pts[:,1]), np.max(pts[:,1])
    pad = 20
    min_x = max(0, min_x-pad); max_x = min(img_w, max_x+pad)
    min_y = max(0, min_y-pad); max_y = min(img_h, max_y+pad)
    bw = max_x-min_x; bh = max_y-min_y
    bcx = (min_x+bw/2)/img_w; bcy = (min_y+bh/2)/img_h
    bwn = bw/img_w; bhn = bh/img_h
    out = f'0 {bcx:.6f} {bcy:.6f} {bwn:.6f} {bhn:.6f} '
    for key in ['femur_shaft','medial_condyle','lateral_condyle','tibia_plateau']:
        px,py = points_2d[key]
        out += f'{min(max(px/img_w,0),1):.6f} {min(max(py/img_h,0),1):.6f} 2 '
    return out.strip()

print('✅ 関数定義完了')

In [ ]:
# データ生成実行
OUT_DIR = '/content/yolo_dataset'
for split in ['train','val']:
    os.makedirs(f'{OUT_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUT_DIR}/labels/{split}', exist_ok=True)

VAL_RATIO  = 0.15
vol_size   = 128
out_size   = (512, 512)
tilts      = [-5, 0, 5]
rots       = [-30, -15, 0, 15, 30]
flexions   = [0, 15, 30, 45]
torsions   = [-10, 0, 10]

volume_normal, lm_normal = create_synthetic_bone_with_landmarks(vol_size, False)
volume_metal,  lm_metal  = create_synthetic_bone_with_landmarks(vol_size, True)
joint_z = int(vol_size * 0.35)

total = len([False,True]) * len(tilts) * len(rots) * len(flexions) * len(torsions) * 2
print(f'生成予定: {total} 枚')

csv_data = []
count = 0

for has_metal in [False, True]:
    volume      = volume_metal  if has_metal else volume_normal
    landmarks_3d= lm_metal      if has_metal else lm_normal
    prefix      = 'metal'       if has_metal else 'bone'

    for t in tilts:
        for r in rots:
            for flex in flexions:
                for torsion in torsions:
                    femur_vol = np.zeros_like(volume)
                    tibia_vol = np.zeros_like(volume)
                    femur_vol[joint_z:,:,:] = volume[joint_z:,:,:]
                    tibia_vol[:joint_z,:,:] = volume[:joint_z,:,:]
                    joint_center = np.array([joint_z, vol_size/2, vol_size/2])
                    kin_mat = get_rotation_matrix(flex, 0, torsion)
                    anat_trans = np.array([-abs(flex)*0.15, 0, 0])
                    off_kin = joint_center - kin_mat.T.dot(joint_center + anat_trans)
                    tibia_moved = affine_transform(tibia_vol, kin_mat.T, offset=off_kin, order=1, mode='constant')

                    for view_type, view_offset in [('LAT',0),('AP',90)]:
                        count += 1
                        name = f'drr_{prefix}_{view_type}_t{t}_r{r}_f{flex}_tor{torsion}'
                        split = 'val' if hash(name) % 100 < (VAL_RATIO*100) else 'train'
                        img_path = f'{OUT_DIR}/images/{split}/{name}.png'
                        lbl_path = f'{OUT_DIR}/labels/{split}/{name}.txt'

                        rot_mat  = get_rotation_matrix(t, r+view_offset, 0)
                        center   = np.array(volume.shape) / 2.0
                        off_glob = center - rot_mat.T.dot(center)

                        demo_seed = int(hashlib.md5(name.encode()).hexdigest(),16) % (2**31)
                        rng = random.Random(demo_seed)
                        bmi = round(rng.uniform(18.0, 35.0), 1)
                        t_score = round(rng.uniform(-3.5, 1.5), 1)

                        femur_rot = affine_transform(femur_vol, rot_mat.T, offset=off_glob, order=1, mode='constant')
                        tibia_rot = affine_transform(tibia_moved, rot_mat.T, offset=off_glob, order=1, mode='constant')
                        proj_raw  = np.clip(np.sum(femur_rot,axis=2) + np.sum(tibia_rot,axis=2), 0, None)
                        proj      = (proj_raw / proj_raw.max() * 255.0) if proj_raw.max() > 0 else proj_raw

                        density = 1.0 - abs(min(t_score,0)) * 0.1
                        proj    = np.clip(proj*density + max(0,(bmi-22)*2.5), 0, 255)
                        drr_img = cv2.resize(proj.astype(np.uint8), out_size, interpolation=cv2.INTER_AREA)
                        cv2.imwrite(img_path, drr_img)

                        pts2d = {}
                        for lname, pt3d in landmarks_3d.items():
                            p = np.array(pt3d, dtype=np.float64)
                            if lname == 'tibia_plateau' or p[0] < joint_z:
                                p = kin_mat.dot(p - joint_center) + joint_center + anat_trans
                            pts2d[lname] = project_3d_point_to_2d(p, rot_mat, center, out_size, volume.shape)
                        with open(lbl_path,'w') as f:
                            f.write(convert_to_yolov8_pose(pts2d, out_size[0], out_size[1]))

                        if count % 100 == 0:
                            print(f'  [{count}/{total}] {name}')

print(f'\n✅ データ生成完了: {count} 枚')

In [ ]:
# dataset.yaml を作成
yaml_content = f"""path: /content/yolo_dataset
train: images/train
val: images/val

kpt_shape: [4, 3]
flip_idx: [0, 2, 1, 3]

names:
  0: knee_joint

# Keypoints:
# 0: femur_shaft
# 1: medial_condyle
# 2: lateral_condyle
# 3: tibia_plateau
"""
with open('/content/yolo_dataset/dataset.yaml','w') as f:
    f.write(yaml_content)

# 生成結果の確認
train_imgs = len(os.listdir('/content/yolo_dataset/images/train'))
val_imgs   = len(os.listdir('/content/yolo_dataset/images/val'))
print(f'Train: {train_imgs} 枚 / Val: {val_imgs} 枚')
print('✅ dataset.yaml 作成完了')

## STEP 4: YOLOv8 Pose モデルを訓練

GPUを使って訓練します。Intel Mac では数時間かかる処理が **数分** で終わります。

In [ ]:
from ultralytics import YOLO
import torch

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（CPUで実行）"}')

# YOLOv8n-pose: 最小サイズ（速い・軽い・まず動かすため）
# 本番は yolov8s-pose や yolov8m-pose に変更する
model = YOLO('yolov8n-pose.pt')

results = model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=50,           # まず50エポック。精度が上がれば100〜200に増やす
    imgsz=512,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/runs',
    name='osteovision_pose',
    exist_ok=True,
    verbose=True
)

print('✅ 訓練完了')

## STEP 5: 訓練済みモデルを Google Drive に保存

In [ ]:
import shutil

best_pt = '/content/runs/osteovision_pose/weights/best.pt'
dest    = f'{SAVE_DIR}/best.pt'

shutil.copy(best_pt, dest)
print(f'✅ モデルを保存しました: {dest}')
print()
print('次のステップ:')
print('1. Google Drive から best.pt をダウンロード')
print('2. OsteoVision_Dev/dicom-viewer-prototype-api/ に配置')
print('3. FastAPI を起動して推論を確認')

## STEP 6: 訓練結果の確認（オプション）

In [ ]:
from IPython.display import Image, display
import glob

# 訓練曲線（精度の推移）
result_img = '/content/runs/osteovision_pose/results.png'
if os.path.exists(result_img):
    display(Image(result_img))

# バリデーション画像（AIが何を検出しているか）
val_imgs = glob.glob('/content/runs/osteovision_pose/val_batch*.jpg')
for img in val_imgs[:2]:
    display(Image(img))